# CPT — Model Training on Google Colab

Trains all 4 learnable models (XGBoost, LightGBM, LSTM, TFT) for **SOL** and **DOGE**.
TimesFM is zero-shot and never trained.

**Before running:**
1. Runtime → Change runtime type → **T4 GPU** (or better)
2. Run all cells top-to-bottom
3. After Cell 13, download the weights from Google Drive → `CPT_models/`
4. Place all downloaded files into your local `models/` directory

**Expected total time on T4:** ~45–60 min for both coins.

---
## 1 — Setup: Mount Drive & Clone Repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

REPO_URL = "https://github.com/its-dreOwO/CPT.git"
REPO_DIR = "/content/CPT"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print("Repo already cloned — pulling latest changes...")
    !cd {REPO_DIR} && git pull

%cd {REPO_DIR}
print("Working directory:", os.getcwd())

---
## 2 — Install Dependencies

Colab already ships with: `torch`, `pandas`, `numpy`, `xgboost`, `lightgbm`, `scikit-learn`.
We only install what's missing.

In [ ]:
!pip install \
    pytorch-forecasting \
    lightning \
    pydantic-settings \
    structlog \
    yfinance \
    sqlalchemy \
    tenacity \
    ccxt \
    --quiet

print("Dependencies installed.")

---
## 3 — Create Minimal `.env`

Training only needs price data — no API keys required.

In [ ]:
env_content = """DATABASE_URL=sqlite:///data/cpt.db
MODEL_DIR=models
REDIS_URL=redis://localhost:6379/0
FRED_API_KEY=placeholder
DISCORD_WEBHOOK_URL=placeholder
TWILIO_ACCOUNT_SID=placeholder
TWILIO_AUTH_TOKEN=placeholder
TWILIO_WHATSAPP_FROM=placeholder
ZALO_OA_ACCESS_TOKEN=placeholder
ZALO_APP_ID=placeholder
ZALO_APP_SECRET=placeholder
ZALO_REFRESH_TOKEN=placeholder
"""

with open(".env", "w") as f:
    f.write(env_content)

print(".env created (price-only mode, no API keys needed for training)")

---
## 4 — Setup Database & Backfill Price Data

Fetches ~730 days of hourly SOL and DOGE OHLCV from Yahoo Finance into SQLite.

In [ ]:
!python scripts/setup_db.py

In [ ]:
!python scripts/backfill_prices.py --coins SOL DOGE --days 730

---
## 5 — Verify GPU

In [ ]:
import torch

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. Training will be slow.")
    print("Go to Runtime > Change runtime type > T4 GPU")

---
## 6 — Train SOL Models

In [ ]:
print("=" * 50)
print("Training SOL — XGBoost")
print("=" * 50)
!python scripts/train_models.py --model xgboost --coin SOL

print()
print("=" * 50)
print("Training SOL — LightGBM")
print("=" * 50)
!python scripts/train_models.py --model lightgbm --coin SOL

In [ ]:
print("=" * 50)
print("Training SOL — LSTM (100 epochs)")
print("=" * 50)
!python scripts/train_models.py --model lstm --coin SOL --epochs 100

In [ ]:
print("=" * 50)
print("Training SOL — TFT (50 epochs)")
print("=" * 50)
!python scripts/train_models.py --model tft --coin SOL --epochs 50

---
## 7 — Train DOGE Models

In [ ]:
print("=" * 50)
print("Training DOGE — XGBoost")
print("=" * 50)
!python scripts/train_models.py --model xgboost --coin DOGE

print()
print("=" * 50)
print("Training DOGE — LightGBM")
print("=" * 50)
!python scripts/train_models.py --model lightgbm --coin DOGE

In [ ]:
print("=" * 50)
print("Training DOGE — LSTM (100 epochs)")
print("=" * 50)
!python scripts/train_models.py --model lstm --coin DOGE --epochs 100

In [ ]:
print("=" * 50)
print("Training DOGE — TFT (50 epochs)")
print("=" * 50)
!python scripts/train_models.py --model tft --coin DOGE --epochs 50

---
## 8 — Evaluate All Models

Target thresholds: **Directional Accuracy > 55%** and **Sharpe Ratio > 1.0**

In [ ]:
print("=" * 50)
print("Evaluating SOL (last 90 days)")
print("=" * 50)
!python scripts/evaluate_models.py --coin SOL --lookback 90

In [ ]:
print("=" * 50)
print("Evaluating DOGE (last 90 days)")
print("=" * 50)
!python scripts/evaluate_models.py --coin DOGE --lookback 90

---
## 9 — List Trained Model Files

In [ ]:
import glob

model_files = sorted(glob.glob("models/*"))
print(f"Found {len(model_files)} model file(s):")
for f in model_files:
    size_mb = os.path.getsize(f) / 1e6
    print(f"  {f}  ({size_mb:.1f} MB)")

---
## 10 — Save Weights to Google Drive

Copies all model files to **Google Drive → `CPT_models/`**.

In [ ]:
import shutil
import glob

DRIVE_SAVE_DIR = "/content/drive/MyDrive/CPT_models"
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)

model_files = glob.glob("models/*")
# Skip .gitkeep
model_files = [f for f in model_files if not f.endswith(".gitkeep")]

for f in model_files:
    dest = os.path.join(DRIVE_SAVE_DIR, os.path.basename(f))
    shutil.copy(f, dest)
    size_mb = os.path.getsize(dest) / 1e6
    print(f"Saved: {os.path.basename(f)}  ({size_mb:.1f} MB)")

print(f"\nAll {len(model_files)} model file(s) saved to Google Drive > CPT_models/")

---
## 11 — Download & Import Locally

1. Open [Google Drive](https://drive.google.com) → find `CPT_models/` folder
2. Select all files → right-click → **Download** (downloads as a zip)
3. Extract the zip
4. Copy all files into your local `models/` directory:
   ```
   CPT/
   └── models/
       ├── xgb_sol_24h_YYYYMMDD.json
       ├── xgb_sol_72h_YYYYMMDD.json
       ├── xgb_sol_168h_YYYYMMDD.json
       ├── lgbm_sol_24h_YYYYMMDD.txt
       ├── lgbm_sol_72h_YYYYMMDD.txt
       ├── lgbm_sol_168h_YYYYMMDD.txt
       ├── lstm_sol_YYYYMMDD.pt
       ├── tft_sol_YYYYMMDD.ckpt
       ├── xgb_doge_*.json  (x3)
       ├── lgbm_doge_*.txt  (x3)
       ├── lstm_doge_YYYYMMDD.pt
       └── tft_doge_YYYYMMDD.ckpt
   ```
5. Run locally to verify:
   ```bash
   python scripts/evaluate_models.py --coin SOL --lookback 90
   python scripts/evaluate_models.py --coin DOGE --lookback 90
   ```